# 13. Readers: exercises

Exercises of [chapter 13](../../13_readers.md). Run the setup cell first.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import json, shutil, tempfile, textwrap
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from data_acquisition.readers import (RECORD_FILES, RECORDS, ckan, entity_bands, files, geotiff, hydroshare, pangaea,
                                      photos, table, vector, zenodo)

READERS = {"zenodo": zenodo, "hydroshare": hydroshare, "ckan": ckan, "pangaea": pangaea}
MD5 = "ab" * 16
RECORD_TEXTS = {        # small records, in the form each repository serves them
    "zenodo": json.dumps({
        "doi": "10.5281/zenodo.2", "conceptdoi": "10.5281/zenodo.1",
        "metadata": {"title": "A survey", "license": {"id": "cc-by-4.0"}, "creators": [{"name": "Rossi, Anna"}],
                     "dates": [{"start": "2019-06-01", "end": "2019-06-30"}], "description": "<p>Flown in June.</p>"},
        "files": [{"key": "ortho.tif", "size": 10, "id": "f1", "checksum": f"md5:{MD5}",
                   "links": {"self": "https://zenodo.org/f/ortho.tif"}}]}),
    "hydroshare": json.dumps({
        "metadata": {"title": "Reach survey", "citation": "Doe (2020). https://doi.org/10.4211/hs.abc",
                     "rights": {"url": "http://creativecommons.org/licenses/by/4.0/", "statement": "CC BY"},
                     "creators": [{"name": "Anna Rossi"}], "abstract": "An abstract",
                     "period_coverage": {"start": "2020-01-01", "end": "2020-12-31"},
                     "spatial_coverage": {"type": "point", "east": -110.5, "north": 44.1}},
        "files": [{"url": "https://x/data/contents/a.csv", "size": 1, "checksum": MD5.upper()},
                  {"url": "https://x/data/contents/b.tif", "size": 2, "checksum": "cd" * 16 + "-2"}]}),
    "ckan": json.dumps({"package": {"result": {
        "id": "pkg-1", "title": "Dams", "license_id": "cc-by", "license_title": "CC BY", "notes": "Some notes",
        "time_period_coverage_start": "0001-01-01", "time_period_coverage_end": "2004-12-07T00:00:00",
        "spatial": json.dumps({"type": "Point", "coordinates": [-85.0, 50.0]}), "organization": {"title": "Ministry"},
        "resources": [{"id": "r1", "format": "ZIP", "url": "https://x/my%20dams.zip", "size": "100", "hash": MD5},
                      {"id": "r2", "format": "HTML", "url": "https://x/viewer"}]}}}),
    "pangaea": ("/* DATA DESCRIPTION:\nCitation:\tRossi, A; Doe, J (2021): Clyst survey [dataset]. PANGAEA, "
                "https://doi.org/10.1594/PANGAEA.1\nLicense:\tCreative Commons Attribution 4.0 (CC-BY-4.0)\n*/\n"
                "Binary\tBinary (Hash)\tBinary (MIME type)\tBinary (Size)\n"
                f"ortho.tif\t{MD5.upper()}\timage/tiff\t1.9 kBytes\nnote.txt\t{MD5}\ttext/plain\t12 Bytes\n"),
}

**1. Rasters.** Read the header of the sample's orthomosaic and of `dem_2022.asc` with `geotiff.info`. Which bands does each declare, and what does the grid lack? Is `README.txt` a raster?

> **From the chapter.** §1: `geotiff.py` reads headers only, and recognises a raster by its extension or by opening it.
>
> **Code.** `geotiff.info`, `geotiff.declared_bands` and `geotiff.is_raster`, in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py).

In [ ]:
...  # your code here (geotiff.info, geotiff.declared_bands, geotiff.is_raster)

**2. Vectors.** Read the extract's `dams.gpkg` (26 July) with `vector.info`. Where did its vertical CRS go, and which geometry type is reported?

> **From the chapter.** §1: `vector.info` returns per layer the geometry, features, fields and CRS, with a compound CRS split.
>
> **Code.** `vector.info`, in [`data_acquisition/readers/vector.py`](../../../../data_acquisition/readers/vector.py).

In [ ]:
...  # your code here

**3. Records.** Read the sample's `metadata.xml` with the reader `RECORDS` gives for it. What does it say about each file's checksum?

> **From the chapter.** §2: one reader per record format, registered in `RECORDS`, and all return the same shape.
>
> **Code.** `RECORD_FILES` and `RECORDS`, in [`data_acquisition/readers/__init__.py`](../../../../data_acquisition/readers/__init__.py); for this file the reader is in [`data_acquisition/readers/eml.py`](../../../../data_acquisition/readers/eml.py).

In [ ]:
...  # your code here (RECORD_FILES, RECORDS[...]["read"])

**4. ArcGIS sidecars.** Read the four `original_*.arcgis.aux.xml` of the extract with `geotiff.arcgis_adjustment`; fit a first-order polynomial through each one's control points and compute the residuals. How do the chunks and the full tiles compare, and what does that say about [chapter 11](../../11_field_data.md)'s alignment results?

> **From the chapter.** §3, and [chapter 3](../../03_drone_surveys.md), §5: ArcGIS keeps its adjustment, with its control points, in a sidecar.
>
> **Code.** `geotiff.arcgis_adjustment`, in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py); `numpy.linalg.lstsq` for the fit.

In [ ]:
aux = EXTRACT / "2018-07-26/original_chunk.arcgis.aux.xml"
x = geotiff.arcgis_adjustment(aux.read_text())
...  # your code here (np.linalg.lstsq on [source, 1] -> target)

**5. Companions.** Which files does `files.companions` give for `ortho.tif` and for `dams.shp`?

> **From the chapter.** §1, and [chapter 1](../../01_geospatial_data.md), §5: companions are decided from file names only.
>
> **Code.** `files.companions`, in [`data_acquisition/readers/files.py`](../../../../data_acquisition/readers/files.py). The files need not exist.

In [ ]:
...  # your code here

**6. Test an edge case.** EXIF stores the altitude as a positive number with a separate flag (`GPSAltitudeRef`: 1 = below sea level). Write a pytest test that makes a photo 400 m below sea level and checks that `photos.exif` reads -400 m. Run it in a scratch folder. What happens?

> **From the chapter.** §4: a reader returns facts about a file. [Chapter 18](../../18_testing.md), §1 explains known bugs and `xfail`.
>
> **Code.** `photos.exif` in [`data_acquisition/readers/photos.py`](../../../../data_acquisition/readers/photos.py), inside a test you write and run with `pytest`; Pillow makes the photo.

In [ ]:
scratch = Path(tempfile.mkdtemp())
...  # write a test to scratch / "test_edge.py" (Pillow: Image.Exif, ExifTags.IFD.GPSInfo), run pytest on it

**7. One shape from five standards.** `RECORD_TEXTS`, in the notebook, holds small records in the Zenodo, HydroShare, CKAN and PANGAEA formats. Read each with its reader: which keys do all share, and which entity did CKAN skip? Which size is only approximate? Then try `ckan.file_name`, `hydroshare.resource_id` and `readers.entity_bands`.

> **From the chapter.** §2: every record reader returns the same shape.
>
> **Code.** `READERS` and `RECORD_TEXTS`, defined above. The readers are in [`data_acquisition/readers/zenodo.py`](../../../../data_acquisition/readers/zenodo.py), [`data_acquisition/readers/hydroshare.py`](../../../../data_acquisition/readers/hydroshare.py), [`data_acquisition/readers/ckan.py`](../../../../data_acquisition/readers/ckan.py) and [`data_acquisition/readers/pangaea.py`](../../../../data_acquisition/readers/pangaea.py); `entity_bands` is in [`data_acquisition/readers/__init__.py`](../../../../data_acquisition/readers/__init__.py).

In [ ]:
...  # your code here (READERS[name].read(RECORD_TEXTS[name]))

**8. Tables.** Write a CSV with a title line, coordinates and a row without coordinates, and one without coordinates at all. What do `table.read`, `table.info` and `table.rows_info` say? What happens if the spec names a column that does not exist?

> **From the chapter.** §1: `table.py` reads CSV and TSV as the summary's `table` spec says.
>
> **Code.** `table.read`, `table.info` and `table.rows_info`, in [`data_acquisition/readers/table.py`](../../../../data_acquisition/readers/table.py).

In [ ]:
scratch = Path(tempfile.mkdtemp())
...  # your code here (spec = {"skip": 1, "x": "Easting", "y": "Northing"})

**9. The small helpers.** `geotiff.sample` on the DSM, with two sizes. `geotiff.unapplied_georeference` on a copy of the extract's chunk, before and after putting its original ArcGIS sidecar next to it; is that sidecar a companion (`files.is_companion`)? `vector.is_vector` and `photos.is_photo_folder` on a few paths; `photos.info` on the sample's photos; `vector.one_layer` on a two-layer GeoPackage.

> **From the chapter.** §1: the helpers of each reader, and §3 for `unapplied_georeference`.
>
> **Code.** `geotiff.sample` and `geotiff.unapplied_georeference` in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py), `files.is_companion` in [`data_acquisition/readers/files.py`](../../../../data_acquisition/readers/files.py), `vector.is_vector` and `vector.one_layer` in [`data_acquisition/readers/vector.py`](../../../../data_acquisition/readers/vector.py), `photos.is_photo_folder` and `photos.info` in [`data_acquisition/readers/photos.py`](../../../../data_acquisition/readers/photos.py).

In [ ]:
scratch = Path(tempfile.mkdtemp())
...  # your code here